# 01 — Feature Engineering

S3 raw reviews + metadata → `feast materialize` (Spark + RAPIDS GPU) → Redis + S3

- `user_features` — per-user aggregates (avg rating, review count, tenure)
- `item_features` — per-item review aggregates (avg rating, stddev, count)
- `item_metadata` — product catalog (title, brand, category, price) from metadata parquet

Features are defined once in `features.py` and materialized to **both** stores:
- **Online (Redis)** — low-latency serving via `get_online_features`
- **Offline (S3 parquet)** — training data via `get_historical_features`

**Prerequisite:** Raw data in MinIO (`envsubst < manifests/data-download-job.yaml | oc apply -f -`)


In [ ]:
%pip install -q boto3 tabulate pandas pyarrow feast redis pyyaml s3fs kubernetes
%pip install yamlmagic --index-url https://pypi.org/simple
%load_ext yamlmagic

## Configuration


In [ ]:
%%yaml parameters

MATERIALIZE_START: "2020-01-01T00:00:00"
MATERIALIZE_END: "2026-12-31T23:59:59"

# Feast repo path inside the Feast pod (git-synced by FeatureStore CR)
FEAST_REPO: /feast-data/smartshop/feast/feature_repo


In [ ]:
from _config import *
globals().update(parameters)

validate()

---
## Pre-flight — Verify Raw Data in S3


In [ ]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    region_name="us-east-1",
)

buckets = [b["Name"] for b in s3.list_buckets()["Buckets"]]
print(f"Existing buckets: {buckets}")

resp = s3.list_objects_v2(Bucket="smartshop-raw", Prefix="raw/reviews/", MaxKeys=5)
raw_files = resp.get("Contents", [])
if raw_files:
    print(f"\n✓ Raw reviews: {len(raw_files)}+ files in smartshop-raw/raw/reviews/")
else:
    print("\n✗ No raw data — run the download manifest first:")
    print("  envsubst < manifests/data-download-job.yaml | oc apply -f -")
    raise RuntimeError("Raw data missing")

expected_cats = ["Electronics", "Books", "Home_and_Kitchen"]
meta_resp = s3.list_objects_v2(Bucket="smartshop-raw", Prefix="raw/metadata/", MaxKeys=50)
meta_keys = [o["Key"] for o in meta_resp.get("Contents", [])]
found_cats = [c for c in expected_cats if any(c in k for k in meta_keys)]
missing_cats = [c for c in expected_cats if c not in found_cats]

print(f"✓ Metadata: {len(meta_keys)} files in smartshop-raw/raw/metadata/")
print(f"  Categories with metadata: {found_cats}")
if missing_cats:
    print(f"  ⚠ Missing metadata for: {missing_cats}")
    print("    Re-run download job with METADATA_ONLY=true to fetch these")

### S3 file listing (optional — expand for detail)


In [ ]:
from tabulate import tabulate

print("=== Reviews ===\n")
paginator = s3.get_paginator("list_objects_v2")
total_size = 0
total_files = 0
rows = []

for page in paginator.paginate(Bucket="smartshop-raw", Prefix="raw/reviews/"):
    for obj in page.get("Contents", []):
        size_mb = obj["Size"] / (1024 * 1024)
        total_size += obj["Size"]
        total_files += 1
        if total_files <= 15:
            rows.append([obj["Key"], f"{size_mb:.1f} MB"])

try:
    print(tabulate(rows, headers=["Key", "Size"], tablefmt="simple"))
except:
    for r in rows:
        print(f"  {r[0]:60s} {r[1]}")

if total_files > 15:
    print(f"  ... and {total_files - 15} more files")
print(f"\nTotal: {total_files} files, {total_size / (1024**3):.2f} GB")

print("\n=== Metadata ===\n")
meta_files = 0
meta_size = 0
for page in paginator.paginate(Bucket="smartshop-raw", Prefix="raw/metadata/"):
    for obj in page.get("Contents", []):
        meta_files += 1
        meta_size += obj["Size"]
        if meta_files <= 10:
            print(f"  {obj['Key']:60s} {obj['Size'] / (1024*1024):.1f} MB")
print(f"\nTotal metadata: {meta_files} files, {meta_size / (1024**3):.2f} GB")

In [ ]:
# Review schema preview
import pandas as pd, io
resp = s3.list_objects_v2(Bucket="smartshop-raw", Prefix="raw/reviews/", MaxKeys=1)
if resp.get("Contents"):
    obj = s3.get_object(Bucket="smartshop-raw", Key=resp["Contents"][0]["Key"])
    df = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    print(f"Schema ({len(df):,} rows): {list(df.columns)}")
    display(df.head(2)) if hasattr(__builtins__, '__IPYTHON__') else print(df.head(2).to_string())

---
## Phase 1 — Register & Materialize

Runs the full Feast pipeline on the Feast pod via `oc exec`:
1. Sync `feature_repo/` to the pod
2. `feast apply` (register feature views)
3. `materialize_incremental` (compute features → Redis + S3 offline parquet)

With `offline=True` on all BatchFeatureViews, materialization writes UDF output
to both Redis (online) and S3 (offline). The S3 offline path is then used by
`get_historical_features` in Phase 3 for training data retrieval.

All operations execute on the pod using the local file registry (avoids gRPC
backtick stripping that affects remote `store.apply()`).

In [ ]:
import time as _time, subprocess, os, tempfile
from kubernetes import client, config
from kubernetes.stream import stream

config.load_incluster_config()
v1 = client.CoreV1Api()

# ── Find the Feast pod ──
feast_pods = v1.list_namespaced_pod(
    NAMESPACE, label_selector="feast.dev/name=smartshop-feast",
    field_selector="status.phase=Running",
).items
assert len(feast_pods) > 0, "No running Feast pods found"
feast_pod = feast_pods[0].metadata.name
print(f"Feast pod: {feast_pod}")

# ── 1. Sync feature_repo to the pod ──
print("\n1. Syncing feature_repo to the Feast pod...")
LOCAL_FEATURE_REPO = os.path.join(os.path.dirname(os.path.abspath("_config.py")), "feature_repo")
for fname in ["features.py", "feature_store.yaml"]:
    src = os.path.join(LOCAL_FEATURE_REPO, fname)
    dst = f"{NAMESPACE}/{feast_pod}:{FEAST_FEATURE_REPO_ON_POD}/{fname}"
    subprocess.run(["oc", "cp", src, dst, "-c", "offline"], check=True)
    print(f"  ✓ {fname}")

# ── 2. feast apply ──
print("\n2. Running feast apply...")
t0 = _time.time()
resp = stream(
    v1.connect_get_namespaced_pod_exec,
    feast_pod, NAMESPACE, container="offline",
    command=["bash", "-c", f"cd {FEAST_FEATURE_REPO_ON_POD} && feast apply 2>&1"],
    stderr=True, stdout=True, stdin=False, _request_timeout=120,
)
elapsed = _time.time() - t0
for line in resp.strip().split("\n")[-10:]:
    print(f"  {line}")
print(f"  ✓ feast apply completed in {elapsed:.0f}s")

# ── 3. materialize_incremental (writes to Redis + S3 offline parquet) ──
print(f"\n3. Running materialize_incremental ({MATERIALIZE_START} → {MATERIALIZE_END})...")
print("   offline=True → features written to both Redis AND S3 offline paths")
MAT_SCRIPT = f"""
import sys, time as _time
sys.path.insert(0, "{FEAST_FEATURE_REPO_ON_POD}")
from datetime import datetime
from feast import FeatureStore

store = FeatureStore(repo_path="{FEAST_FEATURE_REPO_ON_POD}")
t0 = _time.time()
store.materialize_incremental(
    end_date=datetime.fromisoformat("{MATERIALIZE_END}"),
)
elapsed = _time.time() - t0
print(f"Materialized in {{elapsed:.0f}}s", flush=True)
"""
t0 = _time.time()
resp = stream(
    v1.connect_get_namespaced_pod_exec,
    feast_pod, NAMESPACE, container="offline",
    command=["python3", "-u", "-c", MAT_SCRIPT],
    stderr=True, stdout=True, stdin=False, _request_timeout=1800,
)
elapsed = _time.time() - t0
for line in resp.strip().split("\n"):
    if any(kw in line.lower() for kw in ["material", "elapsed", "error", "fail", "view", "redis", "offline"]):
        print(f"  {line}")
print(f"  ✓ materialize_incremental completed in {elapsed:.0f}s")

# ── 4. Verify online + offline stores ──
print("\n4. Verifying...")
import redis
r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT,
                password=REDIS_PASSWORD or None, socket_timeout=5)
keys = r.dbsize()
print(f"  Redis (online): {keys:,} keys")

import boto3
s3 = boto3.client("s3", endpoint_url=S3_ENDPOINT,
                  aws_access_key_id=AWS_KEY, aws_secret_access_key=AWS_SECRET,
                  region_name="us-east-1")
for fv_name in ["user_features", "item_features", "item_metadata"]:
    prefix = f"offline/{fv_name}/"
    resp = s3.list_objects_v2(Bucket=S3_FEATURES_BUCKET, Prefix=prefix, MaxKeys=5)
    n_files = len(resp.get("Contents", []))
    status = "✓" if n_files > 0 else "✗ EMPTY"
    print(f"  S3 offline {fv_name}: {status} ({n_files}+ files)")

from feast import FeatureStore
store = FeatureStore(fs_yaml_file=FEAST_CLIENT_CONFIG)
fvs = store.list_feature_views()
print(f"  Registry: {len(fvs)} feature views")
for fv in fvs:
    offline_flag = getattr(fv, "offline", False)
    print(f"    {fv.name} ({len(fv.features)} features, offline={offline_flag})")

print(f"\n✓ Phase 1 complete — features materialized to Redis + S3 offline")

### 1.2 Verify online features (spot check)

In [ ]:
# Spot-check: fetch a few features from each view via get_online_features
from feast import FeatureStore
store = FeatureStore(fs_yaml_file=FEAST_CLIENT_CONFIG)

sample_user = store.get_online_features(
    features=["user_features:user_avg_rating", "user_features:user_review_count",
              "user_features:user_primary_category"],
    entity_rows=[{"user_id": "AEYORY24FHQJG"}],
).to_dict()

sample_item = store.get_online_features(
    features=["item_features:item_avg_rating", "item_features:item_review_count"],
    entity_rows=[{"item_id": "B09V3KXJPB"}],
).to_dict()

sample_meta = store.get_online_features(
    features=["item_metadata:item_category", "item_metadata:item_title"],
    entity_rows=[{"item_id": "B09V3KXJPB"}],
).to_dict()

print("User features (AEYORY24FHQJG):")
for k, v in sample_user.items():
    if k != "user_id":
        print(f"  {k}: {v[0]}")

print("\nItem features (B09V3KXJPB):")
for k, v in sample_item.items():
    if k != "item_id":
        print(f"  {k}: {v[0]}")

print("\nItem metadata (B09V3KXJPB):")
for k, v in sample_meta.items():
    if k != "item_id":
        print(f"  {k}: {v[0]}")

print("\n✓ All feature views returning data from Redis")

---
## Phase 2 — Verify


In [ ]:
import redis, struct
import pandas as pd

print("Feast registry:")
for fv in store.list_feature_views():
    features = [f.name for f in fv.features] if hasattr(fv, "features") else []
    print(f"  {fv.name:25s} ({len(features)} features)")

r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT, password=REDIS_PASSWORD or None, decode_responses=False)
print(f"\nRedis keys: {r.dbsize():,}")

# Spot-check: verify each feature view has data by scanning a few keys
for entity, fv_name in [("user_id", "user_features"), ("item_id", "item_features"), ("item_id", "item_metadata")]:
    cursor, keys = r.scan(0, match=f"*{entity}*smartshop".encode(), count=50)
    has_fv = 0
    for k in keys[:20]:
        fields = r.hgetall(k)
        if any(fv_name.encode() in f for f in fields.keys()):
            has_fv += 1
    status = "✓" if has_fv > 0 else "✗ NOT FOUND"
    print(f"  {fv_name:25s} → Redis {status} ({has_fv}/{min(20, len(keys))} keys checked)")


### 2.1 User features


In [ ]:
import pandas as pd, struct

_, raw_keys = r.scan(cursor=0, match=b"*user_id*smartshop", count=20)
user_ids = []
for rk in raw_keys:
    raw = rk if isinstance(rk, bytes) else rk.encode()
    pos = raw.find(b"user_id")
    if pos < 0:
        continue
    offset = pos + len(b"user_id") + 4
    if offset + 4 > len(raw):
        continue
    val_len = struct.unpack("<I", raw[offset:offset + 4])[0]
    val = raw[offset + 4 : offset + 4 + val_len].decode("utf-8", errors="ignore")
    if val:
        user_ids.append(val)

sample_users = [{"user_id": uid} for uid in user_ids[:5]]
result = store.get_online_features(
    features=[
        "user_features:user_avg_rating",
        "user_features:user_review_count",
        "user_features:user_unique_items",
        "user_features:user_tenure_days",
        "user_features:user_primary_category",
    ],
    entity_rows=sample_users,
).to_dict()

df = pd.DataFrame(result)
display(df) if hasattr(__builtins__, '__IPYTHON__') else print(df.to_string())

### 2.2 Item features (with product metadata)


In [ ]:
_, raw_keys = r.scan(cursor=0, match=b"*item_id*smartshop", count=200)
item_ids = []
for rk in raw_keys:
    raw = rk if isinstance(rk, bytes) else rk.encode()
    pos = raw.find(b"item_id")
    if pos < 0:
        continue
    offset = pos + len(b"item_id") + 4
    if offset + 4 > len(raw):
        continue
    val_len = struct.unpack("<I", raw[offset:offset + 4])[0]
    val = raw[offset + 4 : offset + 4 + val_len].decode("utf-8", errors="ignore")
    if val:
        item_ids.append(val)

sample_items = [{"item_id": iid} for iid in item_ids[:5]]
result = store.get_online_features(
    features=[
        "item_metadata:item_title",
        "item_metadata:item_brand",
        "item_metadata:item_category",
        "item_features:item_avg_rating",
        "item_metadata:item_price",
        "item_features:item_review_count",
    ],
    entity_rows=sample_items,
).to_dict()

df = pd.DataFrame(result)
has_meta = df["item_title"].notna().sum()
print(f"{has_meta}/{len(df)} items have product metadata")
display(df) if hasattr(__builtins__, '__IPYTHON__') else print(df.to_string())

---
## Phase 3 — Generate Training Data via `get_historical_features`

Uses Feast's `store.get_historical_features()` to produce training data — the **standard**
Feast API for point-in-time correct feature retrieval.

Since `offline=True` was set on all BatchFeatureViews, `materialize_incremental` (Phase 1)
wrote pre-computed features to S3 parquet. `get_historical_features` reads directly from
these offline paths — no UDF re-execution, just a PIT join against the entity DataFrame.

Flow: entity interactions → PIT join with offline S3 features → training parquet.

In [ ]:
HISTORICAL_SCRIPT = f"""
import sys, time as _time, yaml
sys.path.insert(0, "{FEAST_FEATURE_REPO_ON_POD}")
from feast import FeatureStore
from pyspark.sql import SparkSession, functions as F
from pyspark import SparkConf

store = FeatureStore(repo_path="{FEAST_FEATURE_REPO_ON_POD}")

# Create k8s:// distributed SparkSession from batch_engine config.
# get_historical_features reuses the active session for the PIT join.
with open("{FEAST_FEATURE_REPO_ON_POD}/feature_store.yaml") as fh:
    fs_cfg = yaml.safe_load(fh)
batch_cfg = fs_cfg.get("batch_engine", {{}})
spark_pairs = [(k, str(v)) for k, v in batch_cfg.items() if k.startswith("spark.")]
conf = SparkConf().setAll(spark_pairs)
conf.set("spark.app.name", "feast-historical-features")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark.conf.set("spark.sql.runSQLOnFiles", "true")
print(f"SparkSession: {{spark.sparkContext.master}}", flush=True)

# Entity DataFrame: raw interactions.
# event_timestamp = current_timestamp() ensures PIT join matches features
# materialized earlier (whose event_timestamp < now).
print("Building entity DataFrame from raw interactions...", flush=True)
t0 = _time.time()

raw = spark.sql(
    "SELECT user_id, parent_asin AS item_id, rating, "
    "CAST(timestamp / 1000 AS TIMESTAMP) AS event_timestamp "
    "FROM parquet.`s3a://smartshop-raw/raw/reviews/*/` "
    "WHERE rating > 0"
)
raw = raw.withColumn("event_timestamp", F.current_timestamp())
n_raw = raw.count()
print(f"Entity rows: {{n_raw:,}}", flush=True)

# get_historical_features reads from offline S3 paths (pre-computed by
# materialize_incremental with offline=True), NOT from raw data.
print("Calling store.get_historical_features()...", flush=True)
t1 = _time.time()

training_job = store.get_historical_features(
    entity_df=raw.select("user_id", "item_id", "event_timestamp", "rating"),
    features=[
        "user_features:user_avg_rating",
        "user_features:user_review_count",
        "user_features:user_primary_category",
        "item_features:item_avg_rating",
        "item_features:item_review_count",
        "item_metadata:item_category",
    ],
)

training_df = training_job.to_spark_df()
training_df = training_df.withColumn(
    "label", F.when(F.col("rating") >= 4, 1.0).otherwise(0.0)
)

output_path = f"s3a://{S3_FEATURES_BUCKET}/offline/training_data"
training_df.repartition(32).write.mode("overwrite").parquet(output_path)
elapsed = _time.time() - t0
pit_elapsed = _time.time() - t1
n_out = training_df.count()
print(f"Training data: {{n_out:,}} rows in {{elapsed:.0f}}s (PIT join: {{pit_elapsed:.0f}}s)", flush=True)
print(f"Columns: {{training_df.columns}}", flush=True)
spark.stop()
"""

print(f"Generating training data on Feast pod ({feast_pod})...")
print("  get_historical_features: PIT join with offline S3 features → training parquet")
t0 = _time.time()

try:
    resp = stream(
        v1.connect_get_namespaced_pod_exec,
        feast_pod, NAMESPACE, container="offline",
        command=["python3", "-u", "-c", HISTORICAL_SCRIPT],
        stderr=True, stdout=True, stdin=False,
        _request_timeout=3600,
    )
    elapsed = _time.time() - t0
    for line in resp.strip().split("\n"):
        if any(kw in line for kw in ["Entity", "Calling", "Training", "Columns", "Error", "error", "Spark"]):
            print(f"  {line}")

    if "Training data:" in resp:
        print(f"\n✓ Training data generated via get_historical_features in {elapsed:.0f}s")
    else:
        print(f"\n⚠ Check output:")
        for line in resp.strip().split("\n")[-15:]:
            print(f"  {line}")
except Exception as e:
    print(f"✗ Training data generation failed: {e}")

---
## Done

35M+ raw reviews + 2M product metadata:
- **Online features** → Redis via `materialize_incremental` (k8s:// Spark + RAPIDS GPU)
- **Offline features** → S3 parquet via `materialize_incremental` (`offline=True`)
- **Training data** → S3 via `get_historical_features` (PIT join on offline parquet)

Single feature definitions in `features.py` drive both online and offline stores.
`get_historical_features` reads pre-computed features — no UDF re-execution.

**Next →** `02_training.ipynb`
